### setup

In [ ]:
import os
import sys

import numpy as np
import torch

import imfusion

_stdout = sys.stdout
from HD_BET.hd_bet_prediction import get_hdbet_predictor
sys.stdout = _stdout

np.set_printoptions(precision=2, suppress=True)

# Use GPU by default on nvidia prime laptops
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"

os.environ["IMFUSION_PLUGIN_BLACKLIST"]= "Torch;Onnx"

# Rapid prototyping with the ImFusion Python SDK

MICCAI 2026 workshop · September 2026

Ilja Manakov · ImFusion GmbH

<div class="im-agenda"></div>

## What constitutes rapid prototyping?

1. **load** what you need. Regardless of file format.
2. **see** what you are working with. Interactive and in 3D.
3. **process** your data. Powerful algorithms at your fingertips.
4. **leverage** community solutions. Better to integrate then to re-invent whls.
5. **store** everything in one place. Aligned data and metadata. 

---

<div class="im-dark"></div>

## 00

setup

*Up and running in two steps*

<div class="im-dark"></div>

## Getting started

Install the SDK and its base modules from our package index:

```sh
pip install "imfusion-sdk[base]" --extra-index-url https://pypi.imfusion.com/simple
```

Activate your license once, before the first import:

```python
import os
os.environ["IMFUSION_LICENSE_KEY"] = "<your-license-key>"

import imfusion
```

<p class="im-footnote"><a href="https://docs.imfusion.com/python/imfusion-sdk/1.0.0/getting_started/installing.html">Installation guide</a></p>

The activation needs an internet connection. Later imports need no key.

Specialty modules are extras on top of base: `[spine]`, `[ultrasound]`, `[computed-tomography]`, `[vision]`, ...

The free license covers non-commercial use without the specialty modules.

---

<div class="im-blue"></div>

## 01

load

*Different file formats, same loading function.*

## One case, two modalities

- **MR FLAIR**, preoperative — a 192-file DICOM series, as it comes off the scanner
- **3D ultrasound**, intraoperative — NIfTI, as it comes out of a reconstruction pipeline

<p class="im-footnote">Neurosurgical resection case. Data courtesy of SINTEF.</p>

<p><img class="im-contain" src="pictures/case_modalities.png" alt=""></p>

In [ ]:
dcm_files = sorted(os.listdir("data/MR-FLAIR-dicom"))
print(f"Number of slices: {len(dcm_files)}")
print(f"{str(dcm_files[:5])[:-1]}, ...]")

In [ ]:
mr, *_ = imfusion.io.load("data/MR-FLAIR-dicom")

In [ ]:
us, *_ = imfusion.io.load("data/US-intraop.nii.gz")

In [ ]:
ventricles, *_ = imfusion.io.load("data/ventricles_segmentation.mhd")

In [ ]:
print(f"MR: {mr}")
print(f"US: {us}")
print(f"Ventricles: {ventricles}")

## Unified access to metadata

Geometric metadata is directly embedded in each frame of the loaded volume:

In [ ]:
print(f"Spacing: {us[0].spacing}")
print(f"Spacing units are metric (mm)?: {us[0].metric}")
print(f"Image to World Matrix:\n{us[0].image_to_world_matrix}")
print(f"Pixel to World Matrix:\n{us[0].pixel_to_world_matrix}")

## Unified access to metadata

Metadata beyond that is accessible through the DataComponent API:

In [ ]:
print(f"{mr.components.image_info=}")
print(f"{mr.components.general_equipment_module.manufacturer=}")

## Unified access to metadata

Imaging modality is also accessible from the Data instance:

In [ ]:
print(f"Modality of US sweep: {ventricles.modality}")
print(f"Modality of US sweep: {us.modality}")

Nifti does not store modality so we can assign it ourselves. DICOM and MHD, however, do.

In [ ]:
us.modality = imfusion.Data.Modality.ULTRASOUND
print(f"Modality of US sweep: {us.modality}")
print(f"Modality of MRI volume: {mr.modality}")
print(f"Modality of segmentation: {ventricles.modality}")

Knowing modality helps enabling smart presets for downstream processing!

<div class="im-blue"></div>

## Question: 

What are the coordinate conventions for DICOM and Nifti?

<p class="fragment"><img class="im-contain" src="pictures/coordinate_renders_labeled.png" alt=""></p>

## Homogenized coordinate systems

Coordinates are automatically homogenized to LPS on load:

In [ ]:
print("MR image-to-world:\n", mr[0].pixel_to_world_matrix)
print("US image-to-world:\n", us[0].pixel_to_world_matrix)

<p class="im-footnote"><a href="https://docs.imfusion.com/python/coordinatesystems.html">ImFusion conventions for coordinate systems</a></p>

NIfTI is RAS, DICOM is LPS, and the conversion happens on load.

DICOM brought 192 ImagePositionPatient and
ImageOrientationPatient pairs, one per file; the NIfTI brought a single 4x4 affine.

No resampling, changes are purely in the matrix.

 ### We can verify this through SimpleITK and nibabel:

In [ ]:
import nibabel
import SimpleITK as sitk

slice_0 = sitk.ReadImage("data/MR-FLAIR-dicom/IMG_000.dcm")

print("DICOM ImagePositionPatient    :", np.array(slice_0.GetOrigin()))
print("SDK   first MR voxel          :", mr[0].pixel_to_world_matrix[:3, 3])
print("DICOM ImageOrientationPatient :", np.array(slice_0.GetDirection()).reshape(3, 3)[:, :2].T.ravel())
print("SDK   MR image x and y axes   :", mr[0].pixel_to_world_matrix[:3, :2].T.ravel())
print()
print("NIfTI affine origin, RAS      :", nibabel.load("data/US-intraop.nii.gz").affine[:3, 3])
print("SDK   first US voxel, LPS     :", us[0].pixel_to_world_matrix[:3, 3])

<p class="im-footnote"><a href="https://docs.imfusion.com/python/coordinatesystems.html">ImFusion conventions for coordinate systems</a></p>

World frame is the DICOM patient frame.
First MR voxel at (135.8, -136.5, 74.2) mm, matches ImagePositionPatient verbatim
Same for image axes and direction cosines. 
The 192 slice checked for a regular stack and collapsed into one pose.

The NIfTI origin (9.0, -72.5, 15.2) in RAS comes back as (-9.0, 72.5, 15.2) in LPS.

Avoids common left-right flip bugs.

## Why stop at images?

`imfusion.io.load` supports loading Meshes and PointClouds from a variety of file formats:

In [ ]:
mesh, *_ = imfusion.io.load("data/brain_mesh.ply")
pointcloud, *_ = imfusion.io.load("data/brain_pointcloud.pcd")
type(mesh), type(pointcloud)

In [ ]:
len(pointcloud.points)

In [ ]:
mesh.number_of_faces, mesh.number_of_vertices

---

<div class="im-blue"></div>

## 02

see

*No more searching for grey matter in a grey room*

<div class="im-agenda"></div>

## Seeing your data is crucial to understanding it.

- How is the image quality?
- Does it show pathologies?
- How is it aligned?
- How good is my segmentation?

<br><p class="im-lead"><span class="im-accent">imfusion.show</span> to the rescue!</p>

In [ ]:
imfusion.show([mr, us, ventricles])

<p class="im-footnote">More details on <span class="im-accent">imfusion.show</span> in the <a href="https://docs.imfusion.com/python/imfusion-sdk/1.0.0/user_guide/visualization.html">documentation</a></p>

In [ ]:
imfusion.show([mr, mesh, pointcloud])

---

<div class="im-blue"></div>

## 03

process

*Years of R&D effort at your disposal*

## Inter-modal registration the convenient way

With `imfusion.show` we saw that the MR and US volumes were misaligned.

We can fix that in a couple of lines.

### The imfusion.registration module offers powerful registration algorithms

In [ ]:
us_registered = us.clone()

# Create the registration
registration_algo = imfusion.registration.ImageRegistrationAlgorithm(fixed=mr, moving=us_registered)

settings = registration_algo.registration.configuration()
measure = settings["SimilarityMeasureFactory/0"]["mode"].value
print(f"{registration_algo.transformation_model}, similarity measure {measure}")

<p class="im-footnote"><a href="https://scispace.com/pdf/global-registration-of-ultrasound-to-mri-using-the-lc2-7cimt8dawb.pdf>Wein et al. - Global Registration of Ultrasound to MRI Using the LC² Metric</a></p>

The presets of the ImageRegistrationAlgorithm change depending on metadata.
When the modalities are MR and US it picks LC2 as the measure since it is the smarter choice for such a scenario.
If the information is missing it falls back to NCC, which is a reasonable default.
This is an just one example out of many in the ImFusion SDK.

### We just call the algorithm and the registration happens in-place

In [ ]:
registration_algo()

In [ ]:
rotation, translation = imfusion.registration.compute_rigid_pose_distance(
    us[0].image_to_world_matrix,
    us_registered[0].image_to_world_matrix
)
print(
    f"US moved {translation:.1f} mm and was rotated {rotation:.1f} degrees "
    f"in {registration_algo.optimizer.num_eval} evaluations, final {measure} = {registration_algo.best_similarity:.3f}"
)

In [ ]:
imfusion.show([mr, us, us_registered])

## Call any algorithm in the ImFusion SDK

Not all processing capabilites of the ImFusion SDK have bindings to Python yet.

But you can use them from Python anyway!

In [ ]:
imfusion.algorithm.list_available()

We also saw that the MR image has severe bias field artifacts.
We can fix that just as easily:

In [ ]:
imfusion.algorithm.list_available("bias")

In [ ]:
bias_removal = imfusion.algorithm.create("ML.MRIBiasFieldCorrection", [mr])
bias_removal.configuration()

In [ ]:
bias_removal.configure({"iterations": 3})
mr_debiased = bias_removal()[0]
imfusion.show([mr, mr_debiased])

Even when developing in C++ this feature is handy for quickly evaluating your algorithm.

---

<div class="im-blue"></div>

## 04

extend

*Standing on the shoulders of giants*

## Quality control for a retrospective dataset

Say you collected preoperative MR and intraoperative US from a few hundred resections.
MR and US come as separate files, often exported by different tools.

- A **RAS/LPS mix-up** somewhere along the way places the US in the wrong position.
- A **mismatched pair** puts the US of one patient next to the MR of another.

<p><img class="im-contain" src="pictures/quality_control.png" alt=""></p>

## A cheap check for every case

The US was acquired through a craniotomy, so it has to lie inside the brain.

For that we need a brain mask. HD-BET computes one from an MR, so there is no need to train our own.

<p class="im-footnote"><a href="https://github.com/MIC-DKFZ/HD-BET">HD-BET</a>: Isensee et al., Human Brain Mapping, 2019</p>

### ImFusion images convert to and from numpy, so wrapping HD-BET takes only a few lines:

In [ ]:
class BrainExtraction:
    def __init__(self, spacing: float = 2.0):
        self.spacing = spacing
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.predictor = get_hdbet_predictor(device=device)

    def __call__(self, image: imfusion.SharedImageSet) -> imfusion.SharedImageSet:
        # The check does not need a mask at full resolution.
        image = imfusion.processing.resample_to_spacing(image, [self.spacing] * 3)

        volume = image.astype(np.float32).numpy().squeeze()
        mask = self.predictor.predict_single_npy_array(
            volume[None], {"spacing": [self.spacing] * 3}, None, None, False
        )

        # Casting the resampled MR keeps its spacing and pose for the mask.
        brain = image.astype(np.uint8)
        brain[0].assign_array(mask[..., None])
        brain.modality = imfusion.Data.Modality.LABEL
        return brain

At 2 mm HD-BET sees 8x fewer voxels and runs about three times faster, which adds up over a whole dataset. The brain volume changes by less than 0.2 %.

In [ ]:
brain = BrainExtraction(spacing=2.0)(mr)
print(brain)

In [ ]:
imfusion.show([mr, brain])

### Resampling the mask onto each US volume gives the answer:

In [ ]:
inside_brain = {}
for name, volume in (("Original", us), ("Registered", us_registered)):
    field_of_view = volume.numpy() > 0
    mask = imfusion.processing.resample_to_ref_image(
        brain, volume, interpolation_mode=imfusion.InterpolationMode.NEAREST
    ).numpy()
    inside = mask[field_of_view] > 0
    inside_brain[name] = float(inside.mean())
    print(f"{name:<10}: {inside_brain[name]:.1%} of the US field of view is inside the brain")

US voxels with value 0 lie outside the reconstructed fan.
Nearest-neighbor interpolation keeps the mask binary.

For comparison: reading the US NIfTI as LPS instead of RAS leaves about half of the sweep inside the brain.
The header pose is 18.5 mm off and still passes, so this check catches gross errors, not registration accuracy.

---

<div class="im-blue"></div>

## 05

store

*Keep you data and metadata aligned*

## One case, one file

After all this processing, a single case is spread across many files:

- **MR**: 192 DICOM files
- **US**: a NIfTI, with the registration result in a sidecar
- **Ventricles**: an MHD header and its raw data
- **Brain mesh and point cloud**: a PLY and a PCD
- **Brain mask** from HD-BET, and the QC result in a spreadsheet

An ImFusion file holds all of it, including the metadata.

### A data component attaches arbitrary metadata to a Data instance:

In [ ]:
@imfusion.data.components.register(name="quality_control")
class QualityControl:
    pose_change: np.ndarray | None = imfusion.data.components.field()
    lc2: float = imfusion.data.components.field()
    inside_brain: float = imfusion.data.components.field()

    def __init__(self, pose_change: np.ndarray | None = None, lc2: float = 0.0, inside_brain: float = 0.0):
        self.pose_change = pose_change
        self.lc2 = lc2
        self.inside_brain = inside_brain

In [ ]:
us_registered.components.add(
    QualityControl(
        pose_change=us_registered.matrix_to_world(0) @ np.linalg.inv(us.matrix_to_world(0)),
        lc2=registration_algo.best_similarity,
        inside_brain=inside_brain["Registered"],
    )
)
print(us_registered.components.quality_control)

`register` generates serialization, equality and `repr` from the fields.
The name doubles as the attribute on `components`.
numpy arrays such as the pose change are stored natively.

### One call saves the whole case:

In [ ]:
imfusion.io.save([mr, us_registered, ventricles, brain, mesh, pointcloud], "case.imf")

In [ ]:
mr_back, us_back, *_ = imfusion.io.load("case.imf")
print(us_back.components.quality_control)
print(f"Registration kept: {np.allclose(us_back.matrix_to_world(0), us_registered.matrix_to_world(0))}")
print(f"DICOM metadata kept: {mr_back.components.general_equipment_module.manufacturer}")

The registration is stored as the pose of the US, so no transform file is needed.

The component class has to be registered in the reading process too.
Otherwise the SDK logs "Could not factory-create DataComponent" and drops the component.

---

<div class="im-blue"></div>

## 06

see (even more)

*Visual prototyping in the ImFusion Suite (feat. Python)*

## Turning our BrainExtraction into an ImFusion Algorithm

`imfusion.algorithm.register` turns a Python class into an ImFusion algorithm.

It is listed in `imfusion.algorithm.list_available()`, and the ImFusion Suite generates a UI from its parameters.

### The spacing becomes a parameter, the MR an input:

In [ ]:
@imfusion.algorithm.register(display_name="MICCAI;Brain Extraction (HD-BET)")
class BrainExtractionAlgorithm:
    image = imfusion.algorithm.Input(imfusion.SharedImageSet, validator=lambda sis: len(sis) == 1)
    spacing = imfusion.algorithm.ParamDouble("Spacing", default=2.0, min=0.5, max=5.0, unit="mm", with_slider=True)

    def __call__(self) -> imfusion.SharedImageSet:
        return BrainExtraction(self.spacing)(self.image)

In [ ]:
print([name for name in imfusion.algorithm.list_available() if "Brain" in name])
brain_3mm, = imfusion.algorithm.execute("PYTHON.BrainExtractionAlgorithm", [mr], {"Spacing": 3.0})
print(brain_3mm)

Parameters are addressed by their display name.

---

<div class="im-dark"></div>

## ImFusion Suite

Live demo

In [ ]:
%%capture
%%bash
ImFusionSuite brain_extraction_algorithm.py

---

<div class="im-blue"></div>

## 07

process (even more)

*We have only scratched the surface*

<div class="im-agenda-long"></div>

## Specialty modules

1. **Spine** vertebra localization, classification and segmentation in CT and MR, landmark detection in X-ray
2. **Computed tomography** cone-beam simulation, FDK and iterative reconstruction, 2D/3D registration
3. **Ultrasound** sweep compounding, calibration, registration to CT and MR, beamforming
4. **Vision** camera calibration, stereo reconstruction, monocular depth, optical flow
5. **Cranial** skull stripping, whole-brain, tumor and vessel segmentation
6. **Anatomy** anatomical structure collections and statistical shape models

... and many more!

Each module is an extra of `imfusion-sdk`, e.g. `pip install "imfusion-sdk[spine]"`.
The two demos that follow are optional.

### Spine: localization, classification and segmentation in one call

In [ ]:
import imfusion.spine

spine_ct, *_ = imfusion.io.load("data/spine_ct.imf")
spine_ct.modality = imfusion.Data.Modality.CT

labels, spine_data = imfusion.spine.SpineBaseAlgorithm(spine_ct).compute()
print(spine_data)

About 7 s on the GPU. The label map names every vertebra, the sacrum and both hips.

In [ ]:
imfusion.show([spine_ct, labels])

### Each vertebra carries anatomical landmarks, e.g. its pedicle centers:

In [ ]:
pedicle_centers = []
for name in spine_data.vertebrae_names:
    keypoints = spine_data.vertebra(name).keypoints2
    if "ImFusion.Spine.Vertebra.LeftPedicle.Center" in keypoints:
        left = keypoints["ImFusion.Spine.Vertebra.LeftPedicle.Center"]
        right = keypoints["ImFusion.Spine.Vertebra.RightPedicle.Center"]
        pedicle_centers += [left, right]
        print(f"{name:<4}: pedicles {np.linalg.norm(left - right):4.1f} mm apart")

The interpedicular distance widens from the thoracic to the lumbar spine, as expected.
S1 and S2 have no pedicle landmarks.

### Computed tomography: simulate a biplanar X-ray of the same spine:

In [ ]:
import imfusion.computed_tomography as ct

xray = ct.ConeBeamSimulation(
    spine_ct[0],
    geometry_preset=ct.GeometryPreset.BIPLANAR_SHOT,
    proj_type=ct.ProjectionType.LOG_CONVERTED_ATTENUATION,
    frames=2,
)()
print(xray)

### Offset the C-arm pose, then register the CT to the X-ray:

In [ ]:
geometry = ct.ConeBeamMetadata.get(xray).geometry()
true_offset = geometry.recon_offset_z
geometry.recon_rot_x += 4.0  # degrees
geometry.recon_rot_y += 3.0  # degrees
geometry.recon_offset_z += 7.0  # mm
imfusion.show([spine_ct, xray])

In [ ]:
ct.XRay2D3DRegistration(xray, spine_ct, initialization_mode=ct.InitializationMode.NOOP)()
print(
    f"Residual error: {abs(geometry.recon_rot_x):.2f} deg, {abs(geometry.recon_rot_y):.2f} deg, "
    f"{abs(geometry.recon_offset_z - true_offset):.2f} mm"
)

Registration takes under a second.
The offset is recovered exactly because the X-ray was simulated from this CT.
With a real C-arm, the default `InitializationMode.POINT_DIRECTION` provides the starting pose.

In [ ]:
imfusion.show([spine_ct, xray])

<div class="im-blue"></div>

# Thank you

imfusion.com · docs.imfusion.com

manakov@imfusion.com